In [1]:
import numpy as np
import h5py as h5
from multiprocessing import Pool
import torch
import pickle as pk
import sys, os




In [2]:
# arrtest= np.linspace(0, 100, 1000)
# torch.Tensor(arrtest)


In [3]:
# get total number of cpus available:
import os
n_cpus = os.cpu_count()
print(f"Number of cpus available: {n_cpus}")


Number of cpus available: 288


In [4]:
params_all = np.loadtxt('/projects/bdne/spandey3/GOTHAM/prep_data/camels_tng_LH_params.txt', usecols=range(1, 7))
# params.shape
# params[:,5]




In [5]:
norm_delta = 100,
norm_vel = 1000,
BoxSize = 25.
grid = 8
grid_sbox = 32
npart_test = 128**3
nMax_h = 20
nvocab = 64
nrand_sel_box = 64
Mstar_cut = 8

def get_data(isim_fid, subsamp_ds=1):

    params = params_all[isim_fid][None,:]

    sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/LH'
    savefname = f'{sdir}/subhalo_density3Dgrid_{grid_sbox}_isim_{isim_fid}_nrandsubsel_{nrand_sel_box}_nvocab{nvocab}_lgMmin_{Mstar_cut}.pkl'

    saved = pk.load(open(savefname, 'rb'))
    # dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed = saved['dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed']
    # Nhalos_truth_flatten = saved['Nhalos_truth_flatten']
    delta_box_all_squeezed = saved['delta_box_all_squeezed']
    rand_sel = saved['rand_sel']


    sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/LH/wSDSS_photometry'
    savefname = f'{sdir}/subhalo_WITHOUT_density3Dgrid_{grid_sbox}_isim_{isim_fid}_nrandsubsel_{nrand_sel_box}_nvocab{nvocab}_wSDSS_photometry.pkl'
    saved = pk.load(open(savefname, 'rb'))
    dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed = saved['dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed']
    Nhalos_truth_flatten = saved['Nhalos_truth_flatten']


    if subsamp_ds>1:
        nsel = int(len(rand_sel)/subsamp_ds)
        dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed[:nsel,...]
        Nhalos_truth_flatten = Nhalos_truth_flatten[:nsel,...]
        delta_box_all_squeezed = delta_box_all_squeezed[:nsel,...]
        rand_sel = rand_sel[:nsel]

    params_repeated = np.repeat(params, len(rand_sel), axis=0)
    return params_repeated, delta_box_all_squeezed, Nhalos_truth_flatten[:,None], dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed, rand_sel[:,None]




In [6]:
noffset_all = [0, 250, 500, 750]
for nsims_offset in noffset_all:
    print(nsims_offset)
    from tqdm import tqdm
    params_repeated_all = []
    delta_box_all_squeezed_all = []
    Nhalos_truth_flatten_all = []
    dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all = []
    rand_sel_all = []
    # nsims_offset = 0
    nsims_all = nsims_offset + 250
    subsamp_ds = 1
    for ji in tqdm(range(nsims_offset, nsims_all)):
        params_repeated, delta_box_all_squeezed, Nhalos_truth_flatten, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed, rand_sel = get_data(ji, subsamp_ds=subsamp_ds)
        params_repeated_all.append(params_repeated)
        delta_box_all_squeezed_all.append(delta_box_all_squeezed)
        Nhalos_truth_flatten_all.append(Nhalos_truth_flatten)
        dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all.append(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed)
        rand_sel_all.append(rand_sel)


    params_repeated_all = np.concatenate(params_repeated_all, axis=0)
    delta_box_all_squeezed_all = np.concatenate(delta_box_all_squeezed_all, axis=0)
    Nhalos_truth_flatten_all = np.concatenate(Nhalos_truth_flatten_all, axis=0)
    dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all = np.concatenate(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, axis=0)
    rand_sel_all = np.concatenate(rand_sel_all, axis=0)

    isim_fid = 0
    sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/LH/wSDSS_photometry'
    savefname = f'{sdir}/subhalo_WITHOUT_density3Dgrid_{grid_sbox}_isim_{isim_fid}_nrandsubsel_{nrand_sel_box}_nvocab{nvocab}_wSDSS_photometry.pkl'

    saved = pk.load(open(savefname, 'rb'))
    nvocab_total = saved['nvocab_total']
    grid = saved['grid_sbox']
    start_token = saved['start_token']
    pad_token = saved['pad_token']
    end_token = saved['end_token']
    space_token = saved['space_token']
    max_sentence_length = saved['max_sentence_length']

    sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/process_split'
    savefname = f'{sdir}/SIMS_{nsims_offset}_{nsims_all}_data_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_wSDSS_photometry_gri.h5'
    with h5.File(savefname, 'w') as f:
        f.create_dataset('params_repeated_all', data=params_repeated_all)
        f.create_dataset('delta_box_all_squeezed_all', data=delta_box_all_squeezed_all)
        f.create_dataset('Nhalos_truth_flatten_all', data=Nhalos_truth_flatten_all)
        f.create_dataset('dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all', data=dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all)
        f.create_dataset('rand_sel_all', data=rand_sel_all)
        f.create_dataset('nvocab_total', data=nvocab_total)
        f.create_dataset('grid', data=grid)
        f.create_dataset('start_token', data=start_token)
        f.create_dataset('pad_token', data=pad_token)
        f.create_dataset('end_token', data=end_token)
        f.create_dataset('space_token', data=space_token)
        f.create_dataset('max_sentence_length', data=max_sentence_length)
        f.close()




0


100%|██████████| 250/250 [01:48<00:00,  2.30it/s]


250


100%|██████████| 250/250 [01:51<00:00,  2.24it/s]


500


100%|██████████| 250/250 [01:44<00:00,  2.38it/s]


750


100%|██████████| 250/250 [01:59<00:00,  2.09it/s]


In [5]:
def get_data_split(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed, delta_box_all_squeezed, params_all, n1_fac=0.8, n2_fac=0.9, n3_fac=1.0):
    n1 = int(n1_fac*len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed)) 
    n2 = int(n2_fac*len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed)) 
    # n3 = int(n3_fac*len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed))
    train_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed[:n1]
    val_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed[n1:n2]
    # test_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed[n2:n3]

    dm_train = torch.tensor(delta_box_all_squeezed[:n1]).to(torch.float16)
    dm_val = torch.tensor(delta_box_all_squeezed[n1:n2]).to(torch.float16)
    # dm_test = torch.tensor(delta_box_all_squeezed[n2:n3]).to(torch.float16)

    params_all_train = params_all[:n1]
    params_all_val = params_all[n1:n2]
    # params_all_test = params_all[n2:n3]

    x = torch.tensor(train_data_halos[:, :-1])
    y = torch.tensor(train_data_halos[:, 1:])
    mask_train_orig = x != 1
    mask_train = torch.logical_not(mask_train_orig)
    masked_logits = torch.zeros(mask_train.shape)
    mask_train_final = masked_logits.masked_fill(mask_train, float('-inf'))
    mask_train = mask_train_final[:,None,:]
    x, y = torch.tensor(x), torch.tensor(y)
    x_train = x.to(torch.long)
    y_train = y.to(torch.long)
    params_all_train = torch.tensor(params_all_train).to(torch.float16)
    mask_train = torch.tensor(mask_train).to(torch.float16)

    x = torch.tensor(val_data_halos[:, :-1])
    y = torch.tensor(val_data_halos[:, 1:])
    mask_val_orig = x != 1
    mask_val = torch.logical_not(mask_val_orig)
    masked_logits = torch.zeros(mask_val.shape)
    mask_val_final = masked_logits.masked_fill(mask_val, float('-inf'))
    mask_val = mask_val_final[:,None,:]
    x, y = torch.tensor(x), torch.tensor(y)
    x_val = x.to(torch.long)
    y_val = y.to(torch.long)
    params_all_val = torch.tensor(params_all_val).to(torch.float16)
    mask_val = torch.tensor(mask_val).to(torch.float16)

    # x = torch.tensor(test_data_halos[:, :-1])
    # y = torch.tensor(test_data_halos[:, 1:])
    # mask_test_orig = x != 1
    # mask_test = torch.logical_not(mask_test_orig)
    # masked_logits = torch.zeros(mask_test.shape)
    # mask_test_final = masked_logits.masked_fill(mask_test, float('-inf'))
    # mask_test = mask_test_final[:,None,:]
    # x, y = torch.tensor(x), torch.tensor(y)
    # x_test = x.to(torch.long)
    # y_test = y.to(torch.long)
    # params_all_test = torch.tensor(params_all_test).to(torch.float16)
    # mask_test = torch.tensor(mask_test).to(torch.float16)


    return x_train, y_train, dm_train, params_all_train, mask_train, x_val, y_val, dm_val, params_all_val, mask_val



In [6]:
import torch

norm_delta = 100,
norm_vel = 1000,
BoxSize = 25.
grid = 8
grid_sbox = 32
npart_test = 128**3
nMax_h = 20
nvocab = 64
nrand_sel_box = 64
Mstar_cut = 8
subsamp_ds = 1
sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/process_split'
nsims_offset_all = [0, 250, 500, 750]
nsims_all = [250, 250, 250, 250]

# dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, delta_box_all_squeezed_all, params_repeated_all = [], [], []
# delta_box_all_squeezed_all = np.zeros((64000, 30, 32, 32, 32))
Ndevices = 4
sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/process_split'
savefname = f'{sdir}/SPLIT_DATA_{Ndevices}_gpus_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_wSDSS_photometry_gri.h5'
dtype = 'float16'

with h5.File(savefname, 'w') as fo:
    
    
    for ji in range(len(nsims_all)):
        nsims_offset = nsims_offset_all[ji]
        nsims = nsims_all[ji]
        # savefname = f'{sdir}/ALL_data_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_lgMmin_{Mstar_cut}.h5'
        savefname = f'{sdir}/SIMS_{nsims_offset}_{nsims_offset+nsims}_data_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_wSDSS_photometry_gri.h5'    
        # if rank == 0: print(f"Reading data from {savefname}", flush=True)
        # import time
        # if rank == 0:
        with h5.File(savefname, 'r') as f:
            params_repeated_all_ji = f['params_repeated_all'][()]
            print(params_repeated_all_ji.shape)
            # delta_box_all_squeezed_all = f['delta_box_all_squeezed_all'][()]
            # delta_box_all_squeezed_all = np.moveaxis(delta_box_all_squeezed_all, -1, 1)
            dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji = f['dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all'][()]
            print(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji.shape)
            nvocab_total = f['nvocab_total'][()]
            grid_size = f['grid'][()]
            start_token = f['start_token'][()]
            pad_token = f['pad_token'][()]
            end_token = f['end_token'][()]
            max_sentence_length = f['max_sentence_length'][()]    
            f.close()
            
        # if rank == 0:
        def read_hdf5_slice(args):
            file_path, dataset_name, slice_index = args
            with h5.File(file_path, 'r') as f:
                dataset = f[dataset_name]
                data_slice = dataset[slice_index, ...]
            return data_slice[None, ...] 
    
        def load_hdf5_in_parallel(file_path, dataset_name, num_workers=4):
            with h5.File(file_path, 'r') as f:
                dataset = f[dataset_name]
                num_slices = dataset.shape[0]  
                slice_indices = list(range(num_slices))
    
            args = [(file_path, dataset_name, idx) for idx in slice_indices]
    
            with Pool(processes=num_workers) as pool:
                slices = pool.map(read_hdf5_slice, args)
    
            return slices 
    
        dataset_name = "delta_box_all_squeezed_all"
        # num_workers = 30  
        # count the total number of cpus on the machine
        num_workers = 200
        print(f"Number of workers: {num_workers}")
        slices = load_hdf5_in_parallel(savefname, dataset_name, num_workers)
        combined_matrix = np.concatenate(slices, axis=0)  
        delta_box_all_squeezed_all_ji = np.moveaxis(combined_matrix, -1, 1)
    
        x_train, y_train, dm_train, params_train, mask_train, x_val, y_val, dm_val, params_val, mask_val = get_data_split(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji, delta_box_all_squeezed_all_ji, params_repeated_all_ji, 0.8, 0.9, 1.0)
        print(f"Got split with sizes {x_train.shape} and {x_val.shape}", flush=True)        
    
    
        # dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all.append(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all_ji)
        # delta_box_all_squeezed_all.append(delta_box_all_squeezed_all_ji)
        # nsim_per_box = 16000
        
        # delta_box_all_squeezed_all[nsim_per_box*ji:nsim_per_box*(ji+1),...] = delta_box_all_squeezed_all_ji
        # params_repeated_all.append(params_repeated_all_ji)
        rank = ji
        fo.create_dataset(f'x_train_dev_{rank}', data=x_train)
        fo.create_dataset(f'dm_train_dev_{rank}', data=dm_train, dtype=dtype)
        fo.create_dataset(f'params_train_dev_{rank}', data=params_train)
        fo.create_dataset(f'mask_train_dev_{rank}', data=mask_train)
        fo.create_dataset(f'y_train_dev_{rank}', data=y_train)

        fo.create_dataset(f'x_val_dev_{rank}', data=x_val)
        fo.create_dataset(f'dm_val_dev_{rank}', data=dm_val, dtype=dtype)
        fo.create_dataset(f'params_val_dev_{rank}', data=params_val)
        fo.create_dataset(f'mask_val_dev_{rank}', data=mask_val)
        fo.create_dataset(f'y_val_dev_{rank}', data=y_val)

    fo.create_dataset('nvocab_total', data=nvocab_total)
    fo.create_dataset('grid', data=grid_size)
    fo.create_dataset('start_token', data=start_token)
    fo.create_dataset('pad_token', data=pad_token)
    fo.create_dataset('end_token', data=end_token)
    fo.create_dataset('max_sentence_length', data=max_sentence_length)

fo.close()

    


(16000, 6)
(16000, 161)
Number of workers: 200
Got split with sizes torch.Size([12800, 160]) and torch.Size([1600, 160])


/tmp/ipykernel_3594490/1022334939.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x), torch.tensor(y)
/tmp/ipykernel_3594490/1022334939.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask_train = torch.tensor(mask_train).to(torch.float16)
/tmp/ipykernel_3594490/1022334939.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x), torch.tensor(y)
/tmp/ipykernel_3594490/1022334939.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or so

(16000, 6)
(16000, 161)
Number of workers: 200
Got split with sizes torch.Size([12800, 160]) and torch.Size([1600, 160])
(16000, 6)
(16000, 161)
Number of workers: 200
Got split with sizes torch.Size([12800, 160]) and torch.Size([1600, 160])
(16000, 6)
(16000, 161)
Number of workers: 200
Got split with sizes torch.Size([12800, 160]) and torch.Size([1600, 160])


In [ ]:
# collect garbage:
import gc
gc.collect()




In [ ]:
# delta_box_all_squeezed_all = np.concatenate(delta_box_all_squeezed_all, axis=0)
dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all = np.concatenate(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, axis=0)
params_repeated_all = np.concatenate(params_repeated_all, axis=0)




In [ ]:
delta_box_all_squeezed_all.shape, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all.shape, params_repeated_all.shape



In [ ]:
import torch
x_train, y_train, dm_train, params_train, mask_train, x_val, y_val, dm_val, params_val, mask_val = get_data_split(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_all, delta_box_all_squeezed_all, params_repeated_all, 0.8, 1.0)
print(f"Got split with sizes {x_train.shape} and {x_val.shape}", flush=True)        




In [ ]:
int(nrand_sel_box/subsamp_ds)




In [ ]:
# device_id = rank % torch.cuda.device_count()
Ndevices = 4
sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/process_split'
savefname = f'{sdir}/SPLIT_DATA_{Ndevices}_gpus_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_wSDSS_photometry_gri.h5'
dtype = 'float16'

with h5.File(savefname, 'w') as f:

    for rank in range(Ndevices):

        start = rank * (len(x_train) // Ndevices)
        end = start + (len(x_train) // Ndevices)

        x_train_gpu = (x_train[start:end,...])
        dm_train_gpu = (dm_train[start:end,...])
        params_train_gpu = (params_train[start:end,...])
        mask_train_gpu = (mask_train[start:end,...])
        y_train_gpu = (y_train[start:end,...])
        print(x_train_gpu.shape, dm_train_gpu.shape, params_train_gpu.shape, mask_train_gpu.shape, y_train_gpu.shape)
        # print(f"I am rank {rank} and will process train data from {start} to {end}.")
        # if rank == 0: print(f"Transferred train data to GPU", flush=True)        
        f.create_dataset(f'x_train_dev_{rank}', data=x_train_gpu)
        f.create_dataset(f'dm_train_dev_{rank}', data=dm_train_gpu, dtype=dtype)
        f.create_dataset(f'params_train_dev_{rank}', data=params_train_gpu)
        f.create_dataset(f'mask_train_dev_{rank}', data=mask_train_gpu)
        f.create_dataset(f'y_train_dev_{rank}', data=y_train_gpu)


        start = rank * (len(x_val) // Ndevices)
        end = start + (len(x_val) // Ndevices)
        x_val_gpu = (x_val[start:end,...])
        dm_val_gpu = (dm_val[start:end,...])
        params_val_gpu = (params_val[start:end,...])
        mask_val_gpu = (mask_val[start:end,...])
        y_val_gpu = (y_val[start:end,...])
        print(x_val_gpu.shape, dm_val_gpu.shape, params_val_gpu.shape, mask_val_gpu.shape, y_val_gpu.shape)

        f.create_dataset(f'x_val_dev_{rank}', data=x_val_gpu)
        f.create_dataset(f'dm_val_dev_{rank}', data=dm_val_gpu, dtype=dtype)
        f.create_dataset(f'params_val_dev_{rank}', data=params_val_gpu)
        f.create_dataset(f'mask_val_dev_{rank}', data=mask_val_gpu)
        f.create_dataset(f'y_val_dev_{rank}', data=y_val_gpu)

        # print(f"I am rank {rank} and will process val data from {start} to {end}.")    
        # if rank == 0: print(f"Transferred test data to GPU", flush=True)        

    # nvocab_total = f['nvocab_total'][()]
    # grid_size = f['grid'][()]
    # start_token = f['start_token'][()]
    # pad_token = f['pad_token'][()]
    # end_token = f['end_token'][()]
    # max_sentence_length = f['max_sentence_length'][()]    
    f.create_dataset('nvocab_total', data=nvocab_total)
    f.create_dataset('grid', data=grid_size)
    f.create_dataset('start_token', data=start_token)
    f.create_dataset('pad_token', data=pad_token)
    f.create_dataset('end_token', data=end_token)
    f.create_dataset('max_sentence_length', data=max_sentence_length)

f.close()





In [ ]:
norm_delta = 100,
norm_vel = 1000,
BoxSize = 25.
grid = 8
grid_sbox = 32
npart_test = 128**3
nMax_h = 20
nvocab = 64
nrand_sel_box = 64
Mstar_cut = 8
Ndevices = 4
subsamp_ds = 1
sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/process_split'
savefname = f'{sdir}/SPLIT_DATA_{Ndevices}_gpus_nspersim_subhalo_density3Dgrid_{grid_sbox}_isim_all_nrandsubsel_{int(nrand_sel_box/subsamp_ds)}_nvocab{nvocab}_wSDSS_photometry_gri.h5'
dtype = 'float16'
with h5.File(savefname, 'r') as f:
    print(f['x_train_dev_0'][()][-1,:])
        

